# Introduction to Regular Expressions in R

Environmental data rarely arrives clean. Field notes, permit applications, monitoring reports, and web pages all contain information buried inside messy, unstructured text. **Regular expressions** (regex for short) are a mini-language for describing patterns in text — they let you ask questions like:

- *Does this note mention a specific species?*
- *What number comes before the word "hectares"?*
- *How many times does the word "carbon" appear in this report?*

This lecture is a quick, practical introduction. We will cover just enough regex to be useful immediately — and to set you up for the more advanced text analysis tools coming in later lectures.

---

**What we will cover:**

1. Load the `stringr` package — the tidyverse's regex toolkit
2. Detect whether a pattern exists in text (`str_detect`)
3. Extract numbers that appear before a unit (`str_extract`)
4. Count how many times a word appears (`str_count`)
5. Match word variations with character classes and wildcards


# Setup: Load Required Packages

We will use `stringr`, which is part of the tidyverse. It wraps R's built-in regex engine in functions with consistent, readable names — all starting with `str_`.

Having used `dplyr`, `ggplot2`, and other tidyverse packages before, `stringr` should feel familiar.


In [ ]:
# -------------------------------------------------------------------------
# Housekeeping
# -------------------------------------------------------------------------
library(stringr)  # regex toolkit (part of the tidyverse)
library(dplyr)    # data manipulation


# 1. Detecting a Pattern — Does This Text Contain a Word?

The most common regex task is simply asking: **does this string contain a pattern?**

`str_detect(string, pattern)` returns `TRUE` or `FALSE`. That makes it good for creating a new column in a data frame that flags whether a field note, report, or description matches a keyword of interest.

**Our example:** A wildlife biologist has entered field notes into a spreadsheet. We want to automatically flag any note that mentions the northern spotted owl — a threatened species that triggers additional survey requirements.

> **Regex tip:** By default, regex is case-sensitive. We will use `ignore.case = TRUE` inside `regex()` so we catch "Spotted Owl", "spotted owl", "SPOTTED OWL", etc.


In [ ]:
# -------------------------------------------------------------------------
# Sample field notes from a wildlife survey
# -------------------------------------------------------------------------
field_notes <- data.frame(
  survey_id = 1:6,
  notes = c(
    "Observed two northern spotted owls roosting in old-growth Douglas fir stand.",
    "No large mammals detected. Several deer tracks along creek bed.",
    "Spotted Owl heard calling at dusk near ridge line, confirmed by second observer.",
    "Survey transect completed. Beaver activity noted at pond outlet.",
    "Heavy rain limited visibility. No spotted owl detections.",
    "Marbled murrelet nest found. Old-growth canopy intact."
  )
)

field_notes


In [ ]:
# -------------------------------------------------------------------------
# Flag notes that mention "spotted owl" (case-insensitive)
#
# str_detect() returns TRUE/FALSE for each row.
# We use regex(..., ignore_case = TRUE) so capitalization doesn't matter.
# -------------------------------------------------------------------------
field_notes <- field_notes %>%
  mutate(
    spotted_owl_detected = str_detect(notes, regex("spotted owl", ignore_case = TRUE))
  )

field_notes %>% select(survey_id, spotted_owl_detected, notes)


Notice that surveys 1, 3, and 5 are flagged `TRUE`. Survey 5 says "No spotted owl detections" — it still matches because the phrase "spotted owl" appears in the text. That is an important reminder: regex matches the **pattern**, not the meaning. Handling negation ("no X detected") takes more careful pattern design — something to keep in mind as you move into more advanced text analysis.

Now that we have a boolean column, we can filter to only the flagged rows the same way we would filter any other column in `dplyr`:

```r
field_notes %>% filter(spotted_owl_detected == TRUE)
```


# 2. Extracting Numbers Before a Unit

Often we want to pull a specific **value** out of a text string — not just check whether a pattern exists, but actually grab the number. A common real-world case is when data is recorded as prose: *"Harvest totaled 4,200 board feet"* or *"The stand covers 18.5 hectares."*

`str_extract(string, pattern)` returns the first match for a pattern, or `NA` if there is no match.

**The pattern we need:**
- One or more digits: `\\d+`
- Optionally followed by a decimal point and more digits: `(\\.\\d+)?`
- Followed by a space and the unit word

> **Why the double backslash `\\`?**  
> In R strings, `\\` is how you write a single backslash `\`. Regex uses `\d` to mean "any digit". So in R code you write `\\d`.

**Our example:** A state forestry department has stored timber harvest summaries as free-text entries. We want to extract the volume in board feet from each record.


In [ ]:
# -------------------------------------------------------------------------
# Timber harvest summaries stored as free text
# -------------------------------------------------------------------------
harvest_records <- data.frame(
  record_id = 1:5,
  summary = c(
    "Unit 4A harvest completed. Volume: 3200 board feet removed from thinning.",
    "Salvage operation following beetle kill yielded 18500 board feet.",
    "Prescription burn prep: 410 board feet of ladder fuels cut and removed.",
    "No commercial timber harvest. Stand flagged for restoration planting.",
    "Selective harvest in riparian buffer: 750 board feet of dead wood removed."
  )
)

harvest_records


In [ ]:
# -------------------------------------------------------------------------
# Extract the number that appears before "board feet"
#
# Pattern breakdown:
#   \\d+         — one or more digits
#   (\\.\\d+)?   — optionally: a decimal point followed by more digits
#   (?= board feet) — a "lookahead": match only if " board feet" follows
#                     (the lookahead itself is NOT included in the result)
# -------------------------------------------------------------------------
harvest_records <- harvest_records %>%
  mutate(
    board_feet = str_extract(summary, "\\d+(\\.\\d+)?(?= board feet)")
  )

harvest_records


Record 4 returns `NA` — correct, because that record has no board-feet value. The extracted column is still a **character** type (text). If you want to do math with it, convert it to numeric:

```r
harvest_records <- harvest_records %>%
  mutate(board_feet = as.numeric(board_feet))
```

The same pattern works for other units — just swap "board feet" for "hectares", "metric tons", "acres", etc.


# 3. Counting Pattern Occurrences

Sometimes you do not just want to know *whether* a word appears — you want to know *how many times* it appears. This is the foundation of many text analysis methods, including TF-IDF (which you will see in a later lecture).

`str_count(string, pattern)` returns an integer: the number of non-overlapping matches of the pattern in the string.

**Our example:** A researcher has collected summaries of EPA air quality reports from different regions. We want to count how many times the word "wildfire" appears in each summary — a proxy for how smoke-focused the report is.


In [ ]:
# -------------------------------------------------------------------------
# Simulated EPA air quality report summaries
# -------------------------------------------------------------------------
air_reports <- data.frame(
  region = c("Pacific Northwest", "Southwest", "Southeast", "Northeast", "Mountain West"),
  report_text = c(
    "Air quality index exceeded safe levels on 14 days. Wildfire smoke was the primary driver. Wildfire activity has increased each year since 2018. Residents advised to limit outdoor activity during wildfire smoke events.",
    "Ozone levels remained elevated through summer months. No wildfire smoke recorded this season. PM2.5 readings attributed to vehicle and industrial emissions.",
    "Humidity kept particulate matter low. Prescribed burns contributed to one elevated wildfire smoke day in October. Overall air quality rated good.",
    "Wildfire smoke intrusion from western states recorded twice in August. Long-range transport of wildfire particulates is a growing concern for the region.",
    "Wildfire smoke dominated poor air quality days — 22 of 28 unhealthy days were caused by wildfire events. Record drought conditions extended the wildfire season by six weeks."
  )
)

air_reports


In [ ]:
# -------------------------------------------------------------------------
# Count how many times "wildfire" appears in each report (case-insensitive)
# -------------------------------------------------------------------------
air_reports <- air_reports %>%
  mutate(
    wildfire_mentions = str_count(report_text, regex("wildfire", ignore_case = TRUE))
  )

air_reports


The Mountain West and Pacific Northwest have the highest counts, which matches what we know about western wildfire patterns.

`str_count()` is doing word-level counting here. This is exactly the kind of operation that powers **term frequency** in TF-IDF analysis — counting how often each word appears in a document. When you get to that lecture, the pattern will feel familiar.


# 4. Character Classes and Wildcards

So far our patterns have been literal strings. Regex becomes much more powerful when you use **character classes** and **quantifiers** to match families of characters.

| Symbol | Meaning | Example |
|--------|---------|---------|
| `.` | Any single character | `re.est` matches "reforest", "request" |
| `\\d` | Any digit (0–9) | `\\d+` matches "42", "1500" |
| `[abc]` | Any one of a, b, or c | `[aeiou]` matches any vowel |
| `[a-z]` | Any character in a range | `[a-z]+` matches any lowercase word |
| `*` | Zero or more of the previous | `colou*r` matches "color" and "colour" |
| `+` | One or more of the previous | `\\d+` requires at least one digit |
| `?` | Zero or one (makes it optional) | `colou?r` matches "color" and "colour" |
| `^` | Start of string | `^Forest` matches only if text starts with "Forest" |
| `$` | End of string | `Act$` matches only if text ends with "Act" |

**Our example:** A restoration ecologist is searching through a document database for any mention of reforestation activity — but the word appears in several forms: "reforest", "reforested", "reforesting", "reforestation". We want to match all of them with a single pattern.

The trick is to match the common root `reforest` and then allow optional characters after it.


In [ ]:
# -------------------------------------------------------------------------
# Grant abstracts from a forest restoration funding database
# -------------------------------------------------------------------------
grant_abstracts <- data.frame(
  grant_id = 1:6,
  abstract = c(
    "This project will reforest 500 acres of degraded watershed land.",
    "Crews have been reforesting the burned slopes since 2021 with native conifers.",
    "The reforestation effort targets high-severity fire scars across three counties.",
    "Native grassland restoration in the prairie remnant — no tree planting involved.",
    "Phase 2 will reforest an additional 200 acres using locally sourced seedlings.",
    "Carbon sequestration projections assume full reforestation of treated units by 2035."
  )
)

grant_abstracts


In [ ]:
# -------------------------------------------------------------------------
# Detect any form of "reforest" using a wildcard suffix
#
# Pattern: "reforest" followed by zero or more word characters [a-z]*
# This matches: reforest, reforested, reforesting, reforestation
# -------------------------------------------------------------------------
grant_abstracts <- grant_abstracts %>%
  mutate(
    reforest_mention = str_detect(abstract, regex("reforest[a-z]*", ignore_case = TRUE)),
    # Also extract exactly what word was matched
    reforest_word    = str_extract(abstract, regex("reforest[a-z]*", ignore_case = TRUE))
  )

grant_abstracts %>% select(grant_id, reforest_mention, reforest_word)


Grants 1, 2, 3, 5, and 6 all match — each with a different word form. Grant 4 (prairie grassland restoration) correctly returns `FALSE` and `NA` because none of those word forms appear.

The `reforest_word` column shows us exactly which variant appeared in each abstract. This is useful for auditing your pattern and making sure it is not catching unintended matches.

---

**Quick reference — common `stringr` functions:**

| Function | What it does | Returns |
|----------|-------------|---------|
| `str_detect(x, pattern)` | Does the pattern appear? | `TRUE` / `FALSE` |
| `str_extract(x, pattern)` | Pull the first match out | Character (or `NA`) |
| `str_count(x, pattern)` | How many times does it match? | Integer |
| `str_replace(x, pattern, replacement)` | Swap a match for something else | Modified string |

You now have the core toolkit for working with text data in R. In the coming lectures we will use these same ideas — pattern matching, extraction, and counting — inside larger workflows like web scraping, TF-IDF analysis, and reading text from images.
